In [ ]:
# ============================================================
# Richards Equation 1D Finite Difference Solver
# Physics-Based Synthetic Data Augmentation
# For: Physics-Aware Soil Moisture Prediction Project
# Aditya Singh | MS Cybersecurity, Mississippi State University
# Inspired by Boyd et al. (2019)
# ============================================================

# Installs
!pip install scipy -q

# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.integrate import solve_ivp
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Research grade visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 2.0,
})

print("Environment ready ✓")
print("Richards Equation Solver — Phase 1: Parameter Fitting")

In [ ]:
# ============================================================
# Load Real Sensor Data
# ============================================================

print("Upload your plant_vase1(2).csv file...")
uploaded = files.upload()
df_real = pd.read_csv('plant_vase1(2).csv')

print(f"\nReal data loaded ✓")
print(f"Shape: {df_real.shape}")
print(f"Duration: {len(df_real)/60:.1f} hours")
print(f"\nMoisture statistics:")
for col in ['moisture0','moisture1','moisture2','moisture3','moisture4']:
    print(f"  {col}: min={df_real[col].min():.3f} "
          f"max={df_real[col].max():.3f} "
          f"mean={df_real[col].mean():.3f}")

In [ ]:
# ============================================================
# Phase 1: Van Genuchten Model + Parameter Fitting
# Fitting soil hydraulic parameters to real sensor data
# ============================================================

# ---- Van Genuchten Functions ----

def van_genuchten_theta(psi, theta_r, theta_s, alpha, n):
    """
    Van Genuchten soil water retention curve
    psi = matric potential (negative pressure head)
    Returns volumetric water content theta
    """
    m = 1 - 1/n
    Se = 1 / (1 + (alpha * np.abs(psi))**n)**m
    return theta_r + (theta_s - theta_r) * Se

def van_genuchten_K(theta, theta_r, theta_s, alpha, n, Ks):
    """
    Van Genuchten hydraulic conductivity function
    Returns unsaturated hydraulic conductivity K(theta)
    """
    m = 1 - 1/n
    Se = (theta - theta_r) / (theta_s - theta_r)
    Se = np.clip(Se, 1e-6, 1.0)
    K = Ks * Se**0.5 * (1 - (1 - Se**(1/m))**m)**2
    return K

def van_genuchten_psi(theta, theta_r, theta_s, alpha, n):
    """
    Inverse Van Genuchten — theta to psi
    Returns matric potential from water content
    """
    m = 1 - 1/n
    Se = (theta - theta_r) / (theta_s - theta_r)
    Se = np.clip(Se, 1e-6, 1.0 - 1e-6)
    psi = (1/alpha) * (Se**(-1/m) - 1)**(1/n)
    return psi

print("Van Genuchten functions defined ✓")

# ---- 1D Richards Equation Finite Difference Solver ----

def richards_1d_solver(params, t_span, t_eval,
                        theta_init, irrigation_schedule,
                        n_layers=5, dz=0.05):
    """
    1D Richards Equation finite difference solver

    params: [theta_r, theta_s, alpha, n, Ks]
    t_span: (t_start, t_end) in minutes
    t_eval: time points to evaluate at
    theta_init: initial moisture at each layer
    irrigation_schedule: list of (start_min, end_min, rate)
    n_layers: number of soil layers (5 sensors)
    dz: layer thickness in meters
    """
    theta_r, theta_s, alpha, n, Ks = params

    # Clip parameters to physical bounds
    theta_r = np.clip(theta_r, 0.01, 0.15)
    theta_s = np.clip(theta_s, 0.20, 0.60)
    alpha   = np.clip(alpha,   0.01, 1.00)
    n       = np.clip(n,       1.10, 3.00)
    Ks      = np.clip(Ks,      0.001, 0.10)

    params_clipped = [theta_r, theta_s, alpha, n, Ks]

    def richards_rhs(t, theta):
        """Right hand side of Richards Equation"""
        dtheta_dt = np.zeros(n_layers)

        # Get matric potential at each layer
        psi = np.array([van_genuchten_psi(
            th, theta_r, theta_s, alpha, n)
            for th in theta])

        # Get hydraulic conductivity at each layer
        K = np.array([van_genuchten_K(
            th, theta_r, theta_s, alpha, n, Ks)
            for th in theta])

        # Irrigation boundary condition at surface
        q_surface = 0.0
        for irr_start, irr_end, irr_rate in irrigation_schedule:
            if irr_start <= t <= irr_end:
                q_surface = irr_rate
                break

        # Finite difference — interior layers
        for i in range(n_layers):
            # Flux from layer above
            if i == 0:
                # Top boundary — irrigation input
                K_upper = K[0]
                q_in = q_surface
            else:
                K_upper = 0.5 * (K[i] + K[i-1])
                grad_psi_upper = (psi[i] - psi[i-1]) / dz
                q_in = -K_upper * (grad_psi_upper + 1)

            # Flux to layer below
            if i == n_layers - 1:
                # Bottom boundary — free drainage
                q_out = K[i]
            else:
                K_lower = 0.5 * (K[i] + K[i+1])
                grad_psi_lower = (psi[i+1] - psi[i]) / dz
                q_out = -K_lower * (grad_psi_lower + 1)

            # Mass balance
            dtheta_dt[i] = (q_in - q_out) / dz

        return dtheta_dt

    # Solve ODE
    try:
        sol = solve_ivp(
            richards_rhs,
            t_span,
            theta_init,
            t_eval=t_eval,
            method='RK45',
            rtol=1e-4,
            atol=1e-6,
            max_step=5.0
        )

        if sol.success:
            return sol.y.T  # shape: (n_timesteps, n_layers)
        else:
            return None
    except:
        return None

print("Richards Equation 1D solver defined ✓")

# ---- Extract irrigation schedule from real data ----

irrigation_events = []
in_irrigation = False
start_t = 0

for idx, row in df_real.iterrows():
    t_min = idx  # index = minutes from start
    is_irr = row['irrgation'] == True

    if is_irr and not in_irrigation:
        start_t = t_min
        in_irrigation = True
    elif not is_irr and in_irrigation:
        irrigation_events.append((start_t, t_min, 0.002))
        in_irrigation = False

if in_irrigation:
    irrigation_events.append((start_t, len(df_real), 0.002))

print(f"\nIrrigation events detected: {len(irrigation_events)}")
for i, (s, e, r) in enumerate(irrigation_events[:5]):
    print(f"  Event {i+1}: t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min duration)")

# ---- Parameter Fitting ----

print("\nFitting van Genuchten parameters to real data...")
print("This may take 1-2 minutes...")

# Use moisture4 (deepest sensor) as calibration target
# Initial moisture state from real data
theta_init_real = df_real[['moisture0','moisture1',
                            'moisture2','moisture3',
                            'moisture4']].iloc[0].values

# Time vector
t_span = (0, len(df_real)-1)
# Sample every 10 minutes for fitting speed
t_eval_fit = np.arange(0, len(df_real), 10).astype(float)
target_fit = df_real['moisture4'].values[::10]

def objective(params):
    """Minimize RMSE between simulated and real moisture4"""
    result = richards_1d_solver(
        params, t_span, t_eval_fit,
        theta_init_real, irrigation_events
    )

    if result is None:
        return 1000.0

    simulated = result[:, 4]  # moisture4 = deepest layer

    # Handle length mismatch
    min_len = min(len(simulated), len(target_fit))
    rmse = np.sqrt(np.mean(
        (simulated[:min_len] - target_fit[:min_len])**2))

    return rmse

# Initial parameter guess based on data statistics
theta_r_init = df_real['moisture4'].min() * 0.8
theta_s_init = df_real['moisture1'].max() * 0.9
x0 = [theta_r_init, theta_s_init, 0.05, 1.8, 0.01]

print(f"Initial guess: θr={x0[0]:.3f}, θs={x0[1]:.3f}, "
      f"α={x0[2]:.3f}, n={x0[3]:.3f}, Ks={x0[4]:.3f}")

# Bounds for parameters
bounds = [
    (0.01, 0.10),   # theta_r
    (0.20, 0.60),   # theta_s
    (0.01, 0.50),   # alpha
    (1.10, 2.50),   # n
    (0.001, 0.05),  # Ks
]

# Run optimization
result_opt = minimize(
    objective,
    x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 100, 'ftol': 1e-6}
)

fitted_params = result_opt.x
print(f"\nFitted parameters:")
print(f"  θr (residual moisture):        {fitted_params[0]:.4f}")
print(f"  θs (saturated moisture):       {fitted_params[1]:.4f}")
print(f"  α  (air entry):                {fitted_params[2]:.4f}")
print(f"  n  (pore distribution):        {fitted_params[3]:.4f}")
print(f"  Ks (hydraulic conductivity):   {fitted_params[4]:.4f}")
print(f"\nFitting RMSE: {result_opt.fun:.4f} cm³/cm³")
print(f"Optimization success: {result_opt.success}")
print(f"\nParameter fitting complete ✓")

In [ ]:
# ============================================================
# Fix: Correct irrigation column name
# Dataset has typo: 'irrgation' not 'irrigation'
# ============================================================

# Check actual column names
print("Actual columns:", df_real.columns.tolist())
print(f"Irrigation column name: 'irrgation'")
print(f"Irrigation events in data: {df_real['irrgation'].sum()}")

# Re-extract irrigation schedule with correct column name
irrigation_events = []
in_irrigation = False
start_t = 0

for idx, row in df_real.iterrows():
    t_min = idx
    is_irr = row['irrgation'] == True  # correct column name

    if is_irr and not in_irrigation:
        start_t = t_min
        in_irrigation = True
    elif not is_irr and in_irrigation:
        irrigation_events.append((start_t, t_min, 0.002))
        in_irrigation = False

if in_irrigation:
    irrigation_events.append((start_t, len(df_real), 0.002))

print(f"\nIrrigation events detected: {len(irrigation_events)}")
for i, (s, e, r) in enumerate(irrigation_events[:10]):
    print(f"  Event {i+1}: t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min duration)")

In [ ]:
print(df_real['irrgation'].value_counts())
print(df_real['irrgation'].dtype)
print(df_real['irrgation'].head(20))

In [ ]:
# ============================================================
# Fix: Detect irrigation events from moisture spikes
# irrigation column is all False — useless
# Instead detect from surface sensor (moisture0) spikes
# ============================================================

# Irrigation = when surface moisture rises sharply
# Use moisture0 (surface) — most responsive to irrigation

moisture0 = df_real['moisture0'].values

# Calculate rate of change
delta_moisture0 = np.diff(moisture0, prepend=moisture0[0])

# Irrigation event = positive rate of change above threshold
# Looking at the data — moisture0 ranges 0.17 to 0.36
# A rise of > 0.005 per minute indicates irrigation
threshold = 0.005

irrigation_mask = delta_moisture0 > threshold

print(f"Irrigation threshold: {threshold}")
print(f"Timesteps above threshold: {irrigation_mask.sum()}")

# Build irrigation schedule
irrigation_events = []
in_irrigation = False
start_t = 0

for idx in range(len(df_real)):
    if irrigation_mask[idx] and not in_irrigation:
        start_t = idx
        in_irrigation = True
    elif not irrigation_mask[idx] and in_irrigation:
        # Only count events longer than 2 minutes
        if idx - start_t > 2:
            irrigation_events.append((
                float(start_t),
                float(idx),
                0.002
            ))
        in_irrigation = False

if in_irrigation:
    irrigation_events.append((
        float(start_t),
        float(len(df_real)),
        0.002
    ))

print(f"Irrigation events detected: {len(irrigation_events)}")
for i, (s, e, r) in enumerate(irrigation_events):
    print(f"  Event {i+1}: t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min duration) "
          f"hour={df_real.iloc[int(s)]['hour']:02d}:"
          f"{df_real.iloc[int(s)]['minute']:02d}")

# Visualize detected events
fig, ax = plt.subplots(figsize=(16, 5))
time_hours = np.arange(len(df_real)) / 60
ax.plot(time_hours, moisture0,
        color='steelblue', linewidth=1.0, label='moisture0')

for s, e, r in irrigation_events:
    ax.axvspan(s/60, e/60, alpha=0.3,
               color='orange', label='_nolegend_')

ax.axvspan(0, 0, alpha=0.3, color='orange',
           label='Detected irrigation')
ax.set_title('Irrigation Event Detection from Surface Sensor',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Surface Moisture (cm³/cm³)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('irrigation_detection.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fix: Detect irrigation from moisture4 spikes
# moisture4 is cleaner signal — less noise than surface
# ============================================================

moisture4 = df_real['moisture4'].values

# moisture4 baseline is ~0.02, spikes to 0.06-0.11
# Irrigation = moisture4 rises above 0.04
threshold_m4 = 0.04

irrigation_mask = moisture4 > threshold_m4

print(f"Timesteps above threshold: {irrigation_mask.sum()}")

# Build irrigation schedule
irrigation_events = []
in_irrigation = False
start_t = 0

for idx in range(len(df_real)):
    if irrigation_mask[idx] and not in_irrigation:
        start_t = idx
        in_irrigation = True
    elif not irrigation_mask[idx] and in_irrigation:
        if idx - start_t > 5:
            irrigation_events.append((
                float(start_t),
                float(idx),
                0.002
            ))
        in_irrigation = False

if in_irrigation:
    irrigation_events.append((
        float(start_t),
        float(len(df_real)),
        0.002
    ))

print(f"Irrigation events detected: {len(irrigation_events)}")
for i, (s, e, r) in enumerate(irrigation_events):
    print(f"  Event {i+1}: t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min duration) "
          f"Day {df_real.iloc[int(s)]['day']} "
          f"{df_real.iloc[int(s)]['hour']:02d}:"
          f"{df_real.iloc[int(s)]['minute']:02d}")

# Visualize
fig, axes = plt.subplots(2, 1, figsize=(16, 8))
fig.suptitle('Irrigation Event Detection — moisture4 Threshold Method',
             fontsize=13, fontweight='bold')

time_hours = np.arange(len(df_real)) / 60

# Plot moisture4
axes[0].plot(time_hours, moisture4,
             color='steelblue', linewidth=1.0,
             label='moisture4 (deep sensor)')
axes[0].axhline(y=threshold_m4, color='red',
                linestyle='--', linewidth=1.5,
                label=f'Threshold: {threshold_m4}')
for s, e, r in irrigation_events:
    axes[0].axvspan(s/60, e/60, alpha=0.3,
                    color='orange', label='_nolegend_')
axes[0].axvspan(0, 0, alpha=0.3, color='orange',
                label='Detected irrigation')
axes[0].set_title('Deep Sensor (moisture4) with Detection Threshold')
axes[0].set_ylabel('Moisture (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot all sensors with irrigation overlay
for col, color in zip(['moisture0','moisture1','moisture2',
                        'moisture3','moisture4'],
                       ['#e74c3c','#e67e22','#2ecc71',
                        '#3498db','#9b59b6']):
    axes[1].plot(time_hours, df_real[col].values,
                 color=color, linewidth=0.8,
                 label=col, alpha=0.8)

for s, e, r in irrigation_events:
    axes[1].axvspan(s/60, e/60, alpha=0.2,
                    color='orange', label='_nolegend_')

axes[1].set_title('All Sensors with Detected Irrigation Events')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend(fontsize=8, ncol=3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('irrigation_detection_v2.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fix v3: Detect irrigation from moisture1
# moisture1 is clearly the most responsive sensor
# showing dramatic spikes during irrigation
# ============================================================

moisture1 = df_real['moisture1'].values

# moisture1 baseline ~0.35-0.45, spikes to 0.6+
# Use 0.60 as threshold for major irrigation events
threshold_m1 = 0.60

irrigation_mask = moisture1 > threshold_m1

print(f"Threshold: {threshold_m1}")
print(f"Timesteps above threshold: {irrigation_mask.sum()}")

# Build irrigation schedule with merging
# Merge events less than 30 minutes apart
irrigation_events = []
in_irrigation = False
start_t = 0

for idx in range(len(df_real)):
    if irrigation_mask[idx] and not in_irrigation:
        start_t = idx
        in_irrigation = True
    elif not irrigation_mask[idx] and in_irrigation:
        if idx - start_t > 10:
            irrigation_events.append((
                float(start_t),
                float(idx),
                0.003
            ))
        in_irrigation = False

if in_irrigation:
    irrigation_events.append((
        float(start_t),
        float(len(df_real)),
        0.003
    ))

print(f"\nIrrigation events detected: {len(irrigation_events)}")
for i, (s, e, r) in enumerate(irrigation_events):
    print(f"  Event {i+1}: t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min duration) "
          f"Day {df_real.iloc[int(s)]['day']} "
          f"{df_real.iloc[int(s)]['hour']:02d}:"
          f"{df_real.iloc[int(s)]['minute']:02d}")

# Visualize
fig, ax = plt.subplots(figsize=(16, 5))
time_hours = np.arange(len(df_real)) / 60

ax.plot(time_hours, moisture1,
        color='darkorange', linewidth=1.0,
        label='moisture1 (most responsive)')
ax.axhline(y=threshold_m1, color='red',
           linestyle='--', linewidth=1.5,
           label=f'Threshold: {threshold_m1}')

for s, e, r in irrigation_events:
    ax.axvspan(s/60, e/60, alpha=0.3,
               color='orange', label='_nolegend_')
ax.axvspan(0, 0, alpha=0.3, color='orange',
           label='Detected irrigation')

ax.set_title('Irrigation Detection from moisture1\n'
             '(Most Responsive Sensor)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Moisture (cm³/cm³)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('irrigation_detection_v3.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Phase 1 v2: Parameter Fitting with Correct Irrigation Events
# ============================================================

print("Refitting van Genuchten parameters...")
print(f"Using {len(irrigation_events)} irrigation events")
print("This may take 2-3 minutes...\n")

# Use moisture4 as calibration target — deepest sensor
# most physically meaningful for Richards Equation
theta_init_real = df_real[['moisture0','moisture1',
                            'moisture2','moisture3',
                            'moisture4']].iloc[0].values.copy()

# Clip initial conditions to physical bounds
theta_init_real = np.clip(theta_init_real, 0.01, 0.55)

print(f"Initial moisture state: {theta_init_real}")

# Time vector — sample every 10 minutes for speed
t_span = (0.0, float(len(df_real)-1))
t_eval_fit = np.arange(0, len(df_real), 10).astype(float)
target_fit = df_real['moisture4'].values[::10]

print(f"Calibration target: moisture4")
print(f"Fitting on {len(t_eval_fit)} sample points")

def objective_v2(params):
    """Minimize RMSE between simulated and real moisture4"""
    result = richards_1d_solver(
        params, t_span, t_eval_fit,
        theta_init_real.copy(), irrigation_events
    )
    if result is None:
        return 1000.0
    simulated = result[:, 4]
    min_len = min(len(simulated), len(target_fit))
    if min_len == 0:
        return 1000.0
    rmse = np.sqrt(np.mean(
        (simulated[:min_len] - target_fit[:min_len])**2))
    return rmse if np.isfinite(rmse) else 1000.0

# Better initial guess based on actual data statistics
x0 = [
    float(df_real['moisture4'].min()),      # theta_r
    float(df_real['moisture1'].quantile(0.95)) * 0.8,  # theta_s
    0.08,   # alpha
    1.60,   # n
    0.008   # Ks
]

print(f"\nInitial guess:")
print(f"  θr={x0[0]:.4f}, θs={x0[1]:.4f}, "
      f"α={x0[2]:.4f}, n={x0[3]:.4f}, Ks={x0[4]:.4f}")

bounds = [
    (0.005, 0.08),   # theta_r
    (0.15,  0.55),   # theta_s
    (0.01,  0.40),   # alpha
    (1.10,  2.50),   # n
    (0.001, 0.05),   # Ks
]

# First pass — coarse optimization
result_opt = minimize(
    objective_v2, x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 200, 'ftol': 1e-8}
)

fitted_params = result_opt.x

print(f"\nFitted parameters:")
print(f"  θr (residual moisture):      {fitted_params[0]:.4f}")
print(f"  θs (saturated moisture):     {fitted_params[1]:.4f}")
print(f"  α  (air entry):              {fitted_params[2]:.4f}")
print(f"  n  (pore distribution):      {fitted_params[3]:.4f}")
print(f"  Ks (hydraulic conductivity): {fitted_params[4]:.4f}")
print(f"\nFitting RMSE: {result_opt.fun:.6f} cm³/cm³")
print(f"Optimization success: {result_opt.success}")

# Quick validation — run solver with fitted params
print("\nValidating fitted parameters...")
t_eval_full = np.arange(0, len(df_real), 1).astype(float)
sim_result = richards_1d_solver(
    fitted_params, t_span, t_eval_full,
    theta_init_real.copy(), irrigation_events
)

if sim_result is not None:
    print(f"Solver ran successfully ✓")
    print(f"Output shape: {sim_result.shape}")
    print(f"\nSimulated moisture4 stats:")
    print(f"  Mean: {sim_result[:,4].mean():.4f}")
    print(f"  Std:  {sim_result[:,4].std():.4f}")
    print(f"  Min:  {sim_result[:,4].min():.4f}")
    print(f"  Max:  {sim_result[:,4].max():.4f}")
    print(f"\nReal moisture4 stats:")
    print(f"  Mean: {df_real['moisture4'].mean():.4f}")
    print(f"  Std:  {df_real['moisture4'].std():.4f}")
    print(f"  Min:  {df_real['moisture4'].min():.4f}")
    print(f"  Max:  {df_real['moisture4'].max():.4f}")
else:
    print("Solver failed — parameters need adjustment")

In [ ]:
# ============================================================
# Stable Richards Equation Solver
# Using explicit finite difference with adaptive timestep
# and numerical stability checks
# ============================================================

def richards_stable_solver(params, n_minutes,
                             irrigation_schedule,
                             theta_init,
                             n_layers=5,
                             dz=0.05,
                             dt=0.1):
    """
    Stable explicit finite difference solver
    Uses small fixed timestep dt (minutes) for stability
    Returns moisture at each layer for each minute
    """
    theta_r, theta_s, alpha, n, Ks = params

    # Clip to physical bounds
    theta_r = np.clip(theta_r, 0.005, 0.10)
    theta_s = np.clip(theta_s, 0.15,  0.60)
    alpha   = np.clip(alpha,   0.005, 0.50)
    n       = np.clip(n,       1.10,  3.00)
    Ks      = np.clip(Ks,      0.0001,0.10)
    m       = 1.0 - 1.0/n

    def Se(theta):
        """Effective saturation"""
        s = (theta - theta_r) / (theta_s - theta_r)
        return np.clip(s, 1e-6, 1.0 - 1e-6)

    def K_func(theta):
        """Hydraulic conductivity"""
        se = Se(theta)
        return Ks * se**0.5 * (1-(1-se**(1/m))**m)**2

    def C_func(theta):
        """Specific moisture capacity dtheta/dpsi"""
        se = Se(theta)
        psi = (1/alpha) * (se**(-1/m) - 1)**(1/n)
        psi = np.clip(np.abs(psi), 1e-6, 1e6)
        C = (alpha**n * n * m * (theta_s - theta_r) *
             psi**(n-1)) / (1 + (alpha*psi)**n)**(m+1)
        return np.clip(C, 1e-8, 10.0)

    # Initialize
    theta = theta_init.copy().astype(float)
    theta = np.clip(theta, theta_r + 1e-4,
                    theta_s - 1e-4)

    # Storage — save every minute
    n_steps = int(n_minutes / dt)
    save_every = int(1.0 / dt)
    results = np.zeros((n_minutes, n_layers))
    results[0] = theta.copy()

    save_idx = 1

    for step in range(1, n_steps):
        t_min = step * dt

        # Get hydraulic conductivity
        K = K_func(theta)

        # Surface boundary — irrigation flux
        q_top = 0.0
        for irr_start, irr_end, irr_rate in irrigation_schedule:
            if irr_start <= t_min <= irr_end:
                q_top = irr_rate
                break

        dtheta = np.zeros(n_layers)

        for i in range(n_layers):
            # Inter-nodal conductivities
            # (geometric mean for stability)
            if i == 0:
                K_up = K[0]
                q_in = q_top
            else:
                K_up = np.sqrt(K[i] * K[i-1])
                dpsi_up = ((theta[i] - theta[i-1]) /
                           (C_func(theta[i]) * dz))
                q_in = -K_up * (dpsi_up + 1.0)

            if i == n_layers - 1:
                # Free drainage at bottom
                q_out = K[i]
            else:
                K_down = np.sqrt(K[i] * K[i+1])
                dpsi_down = ((theta[i+1] - theta[i]) /
                             (C_func(theta[i]) * dz))
                q_out = -K_down * (dpsi_down + 1.0)

            dtheta[i] = dt * (q_in - q_out) / dz

        # Update theta
        theta = theta + dtheta

        # Enforce physical bounds
        theta = np.clip(theta, theta_r + 1e-4,
                        theta_s - 1e-4)

        # Check stability
        if not np.all(np.isfinite(theta)):
            print(f"  Stability issue at t={t_min:.1f} min")
            theta = np.clip(theta_init.copy(),
                           theta_r + 1e-4,
                           theta_s - 1e-4)

        # Save every minute
        if step % save_every == 0 and save_idx < n_minutes:
            results[save_idx] = theta.copy()
            save_idx += 1

    return results

print("Stable Richards solver defined ✓")

# ============================================================
# Test solver with initial parameter guess
# ============================================================

print("\nTesting stable solver...")

# Simple initial guess
test_params = [0.01, 0.45, 0.05, 1.5, 0.005]

theta_init = df_real[['moisture0','moisture1',
                       'moisture2','moisture3',
                       'moisture4']].iloc[0].values.copy()
theta_init = np.clip(theta_init, 0.02, 0.50)

test_result = richards_stable_solver(
    test_params,
    n_minutes=len(df_real),
    irrigation_schedule=irrigation_events,
    theta_init=theta_init
)

print(f"Solver output shape: {test_result.shape}")
print(f"\nSimulated moisture4:")
print(f"  Mean: {test_result[:,4].mean():.4f}")
print(f"  Std:  {test_result[:,4].std():.4f}")
print(f"  Min:  {test_result[:,4].min():.4f}")
print(f"  Max:  {test_result[:,4].max():.4f}")
print(f"\nReal moisture4:")
print(f"  Mean: {df_real['moisture4'].mean():.4f}")
print(f"  Std:  {df_real['moisture4'].std():.4f}")
print(f"  Min:  {df_real['moisture4'].min():.4f}")
print(f"  Max:  {df_real['moisture4'].max():.4f}")

# Quick visualization
fig, ax = plt.subplots(figsize=(14, 5))
time_hours = np.arange(len(df_real)) / 60
ax.plot(time_hours, df_real['moisture4'].values,
        color='black', linewidth=1.0,
        label='Real moisture4', alpha=0.8)
ax.plot(time_hours, test_result[:,4],
        color='steelblue', linewidth=1.5,
        linestyle='--', label='Simulated moisture4')
ax.set_title('Stable Solver Test — Initial Parameters',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Time (hours)')
ax.set_ylabel('Moisture (cm³/cm³)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('solver_test.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Fast Parameter Estimation
# Analytical approach — no expensive solver calls
# ============================================================

print("="*55)
print("FAST PARAMETER ESTIMATION")
print("="*55)

moisture4_real = df_real['moisture4'].values

# Extract key statistics from real data
theta_min = moisture4_real.min()
theta_max = moisture4_real.max()
theta_mean = moisture4_real.mean()

print(f"Real moisture4 statistics:")
print(f"  Min (≈ θr): {theta_min:.4f}")
print(f"  Max (≈ θs): {theta_max:.4f}")
print(f"  Mean:       {theta_mean:.4f}")

# Direct parameter estimation from data statistics
# θr ≈ minimum moisture (residual)
# θs ≈ maximum moisture (saturated)
theta_r_est = theta_min * 0.9
theta_s_est = theta_max * 1.1

# Estimate Ks from drainage rate
# Find post-irrigation drainage periods
# Rate of decrease in moisture4 after irrigation
drainage_rates = []
for s, e, r in irrigation_events:
    end_idx = int(e)
    if end_idx + 60 < len(moisture4_real):
        # Moisture at end of irrigation
        m_start = moisture4_real[end_idx]
        # Moisture 60 minutes later
        m_end = moisture4_real[end_idx + 60]
        if m_start > m_end:
            rate = (m_start - m_end) / 60
            drainage_rates.append(rate)

if drainage_rates:
    mean_drainage = np.mean(drainage_rates)
    Ks_est = mean_drainage * 0.05
else:
    Ks_est = 0.005

print(f"\nEstimated parameters from data:")
print(f"  θr = {theta_r_est:.4f}")
print(f"  θs = {theta_s_est:.4f}")
print(f"  Ks = {Ks_est:.6f} (from drainage rate)")

# Use fixed physically reasonable values for alpha and n
# These are typical for loamy/potting soil
alpha_est = 0.05
n_est = 1.40

fitted_params = np.array([theta_r_est, theta_s_est,
                            alpha_est, n_est, Ks_est])

print(f"\nFinal estimated parameters:")
print(f"  θr (residual):           {fitted_params[0]:.4f}")
print(f"  θs (saturated):          {fitted_params[1]:.4f}")
print(f"  α  (air entry):          {fitted_params[2]:.4f}")
print(f"  n  (pore distribution):  {fitted_params[3]:.4f}")
print(f"  Ks (conductivity):       {fitted_params[4]:.6f}")

# Run single validation with estimated params
print(f"\nRunning validation simulation...")
theta_init_val = df_real[['moisture0','moisture1',
                           'moisture2','moisture3',
                           'moisture4']].iloc[0].values.copy()
theta_init_val = np.clip(theta_init_val,
                          fitted_params[0] + 0.001,
                          fitted_params[1] - 0.001)

val_result = richards_stable_solver(
    fitted_params,
    n_minutes=len(df_real),
    irrigation_schedule=irrigation_events,
    theta_init=theta_init_val
)

rmse_val = np.sqrt(np.mean(
    (val_result[:,4] - moisture4_real)**2))

print(f"\nValidation RMSE: {rmse_val:.6f} cm³/cm³")
print(f"\nSimulated vs Real moisture4:")
print(f"  Mean: {val_result[:,4].mean():.4f} vs "
      f"{moisture4_real.mean():.4f}")
print(f"  Std:  {val_result[:,4].std():.4f} vs "
      f"{moisture4_real.std():.4f}")
print(f"  Min:  {val_result[:,4].min():.4f} vs "
      f"{moisture4_real.min():.4f}")
print(f"  Max:  {val_result[:,4].max():.4f} vs "
      f"{moisture4_real.max():.4f}")

# Physical Validation Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Physical Validation: Simulation vs Observations\n'
             'Richards Equation with Data-Fitted Van Genuchten Parameters',
             fontsize=13, fontweight='bold')

time_hours = np.arange(len(df_real)) / 60
layer_colors = ['#e74c3c','#e67e22','#2ecc71',
                '#3498db','#9b59b6']
layer_names = ['moisture0','moisture1','moisture2',
               'moisture3','moisture4']

# Plot 1: moisture4 focus
axes[0].plot(time_hours, moisture4_real,
             color='black', linewidth=1.2,
             label='Observed moisture4', zorder=5)
axes[0].plot(time_hours, val_result[:,4],
             color='steelblue', linewidth=2.0,
             linestyle='--',
             label=f'Simulated (RMSE={rmse_val:.4f})',
             zorder=4)
for s, e, r in irrigation_events:
    axes[0].axvspan(s/60, e/60, alpha=0.15,
                    color='orange', label='_nolegend_')
axes[0].axvspan(0, 0, alpha=0.15, color='orange',
                label='Irrigation event')
axes[0].set_title('Deep Sensor (moisture4) — '
                  'Simulation vs Observation',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('Volumetric Water Content (cm³/cm³)')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: All layers
for i, (col, color) in enumerate(zip(layer_names,
                                      layer_colors)):
    axes[1].plot(time_hours, df_real[col].values,
                 color=color, linewidth=1.0,
                 label=f'Obs {col}', alpha=0.8)
    axes[1].plot(time_hours, val_result[:,i],
                 color=color, linewidth=1.5,
                 linestyle='--', alpha=0.6,
                 label=f'Sim {col}')

for s, e, r in irrigation_events:
    axes[1].axvspan(s/60, e/60, alpha=0.1,
                    color='orange')

axes[1].set_title('All Sensor Layers — '
                  'Simulation vs Observation',
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Volumetric Water Content (cm³/cm³)')
axes[1].legend(fontsize=7, ncol=5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('physical_validation.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nPhysical validation plot saved ✓")
print("Parameter estimation complete ✓")

In [ ]:
# ============================================================
# Phase 1 Final: Parameter Fitting with Stable Solver
# ============================================================

print("="*55)
print("PARAMETER FITTING — STABLE SOLVER")
print("="*55)

theta_init_fit = df_real[['moisture0','moisture1',
                           'moisture2','moisture3',
                           'moisture4']].iloc[0].values.copy()
theta_init_fit = np.clip(theta_init_fit, 0.02, 0.50)

# Sample every 5 minutes for fitting
sample_rate = 5
t_indices = np.arange(0, len(df_real), sample_rate)
target_m4 = df_real['moisture4'].values[t_indices]

def objective_stable(params):
    """RMSE between simulated and real moisture4"""
    try:
        result = richards_stable_solver(
            params,
            n_minutes=len(df_real),
            irrigation_schedule=irrigation_events,
            theta_init=theta_init_fit.copy()
        )
        simulated = result[t_indices, 4]
        min_len = min(len(simulated), len(target_m4))
        rmse = np.sqrt(np.mean(
            (simulated[:min_len] -
             target_m4[:min_len])**2))
        return rmse if np.isfinite(rmse) else 10.0
    except:
        return 10.0

# Better initial guess — force low theta_s
# Real data shows moisture4 max = 0.11
# So theta_s should be around 0.12-0.15
x0 = [0.010,  # theta_r — close to real min (0.01)
      0.120,  # theta_s — close to real max (0.11)
      0.080,  # alpha
      1.500,  # n
      0.020]  # Ks — higher for faster drainage

print(f"Initial guess:")
print(f"  θr={x0[0]:.4f}, θs={x0[1]:.4f}, "
      f"α={x0[2]:.4f}, n={x0[3]:.4f}, Ks={x0[4]:.4f}")
print(f"Initial RMSE: {objective_stable(x0):.6f}")

bounds = [
    (0.005, 0.015),  # theta_r — tight around real min
    (0.080, 0.150),  # theta_s — tight around real max
    (0.010, 0.500),  # alpha
    (1.100, 2.500),  # n
    (0.005, 0.100),  # Ks — allow faster drainage
]

print(f"\nRunning optimization...")
print(f"(fitting to {len(t_indices)} sample points)\n")

result_opt = minimize(
    objective_stable,
    x0,
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 300, 'ftol': 1e-10}
)

fitted_params = result_opt.x

print(f"Fitted parameters:")
print(f"  θr (residual moisture):      {fitted_params[0]:.4f}")
print(f"  θs (saturated moisture):     {fitted_params[1]:.4f}")
print(f"  α  (air entry):              {fitted_params[2]:.4f}")
print(f"  n  (pore distribution):      {fitted_params[3]:.4f}")
print(f"  Ks (hydraulic conductivity): {fitted_params[4]:.4f}")
print(f"\nFitting RMSE: {result_opt.fun:.6f} cm³/cm³")
print(f"Optimization success: {result_opt.success}")

# Validate fitted parameters
print("\nValidating fitted parameters...")
val_result = richards_stable_solver(
    fitted_params,
    n_minutes=len(df_real),
    irrigation_schedule=irrigation_events,
    theta_init=theta_init_fit.copy()
)

print(f"\nSimulated moisture4 stats:")
print(f"  Mean: {val_result[:,4].mean():.4f} "
      f"(real: {df_real['moisture4'].mean():.4f})")
print(f"  Std:  {val_result[:,4].std():.4f} "
      f"(real: {df_real['moisture4'].std():.4f})")
print(f"  Min:  {val_result[:,4].min():.4f} "
      f"(real: {df_real['moisture4'].min():.4f})")
print(f"  Max:  {val_result[:,4].max():.4f} "
      f"(real: {df_real['moisture4'].max():.4f})")

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Richards Equation Parameter Fitting Results\n'
             'Van Genuchten Parameters Calibrated to Real Data',
             fontsize=13, fontweight='bold')

time_hours = np.arange(len(df_real)) / 60

# Plot 1: moisture4 comparison
axes[0].plot(time_hours, df_real['moisture4'].values,
             color='black', linewidth=1.0,
             label='Real moisture4', alpha=0.8)
axes[0].plot(time_hours, val_result[:,4],
             color='steelblue', linewidth=1.5,
             linestyle='--', label='Fitted simulation')
for s, e, r in irrigation_events:
    axes[0].axvspan(s/60, e/60, alpha=0.15,
                    color='orange', label='_nolegend_')
axes[0].axvspan(0, 0, alpha=0.15, color='orange',
                label='Irrigation event')
axes[0].set_title(f'Deep Sensor Calibration — '
                  f'RMSE: {result_opt.fun:.4f} cm³/cm³',
                  fontsize=11, fontweight='bold')
axes[0].set_ylabel('Moisture (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: All layers comparison
layer_colors = ['#e74c3c','#e67e22','#2ecc71',
                '#3498db','#9b59b6']
layer_names = ['moisture0','moisture1','moisture2',
               'moisture3','moisture4']

for i, (col, color) in enumerate(zip(layer_names,
                                      layer_colors)):
    axes[1].plot(time_hours,
                 df_real[col].values,
                 color=color, linewidth=0.8,
                 label=f'Real {col}', alpha=0.7)
    axes[1].plot(time_hours,
                 val_result[:,i],
                 color=color, linewidth=1.5,
                 linestyle='--', alpha=0.9,
                 label=f'Sim {col}')

axes[1].set_title('All Layers — Real vs Simulated',
                   fontsize=11, fontweight='bold')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend(fontsize=7, ncol=5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('parameter_fitting_results.png',
            dpi=150, bbox_inches='tight')
plt.show()

print("\nParameter fitting complete ✓")
print(f"Fitted params saved for synthetic generation")

In [ ]:
# ============================================================
# Phase 2: Synthetic Data Generation
# Using fitted parameters to generate 30 days of data
# Focus: realistic irrigation cycles for moisture4
# ============================================================

print("="*55)
print("PHASE 2: SYNTHETIC DATA GENERATION")
print("="*55)

import random
random.seed(42)
np.random.seed(42)

# Adjusted parameters — tune for better moisture4 match
# Key insight: reduce Ks to slow drainage
# Increase alpha to get sharper wetting front
adjusted_params = np.array([
    0.009,   # theta_r — keep close to real min
    0.115,   # theta_s — just above real max
    0.080,   # alpha — increased for sharper response
    1.600,   # n — slightly higher
    0.002    # Ks — reduced for slower drainage
])

# Validate adjusted params first
val_adj = richards_stable_solver(
    adjusted_params,
    n_minutes=len(df_real),
    irrigation_schedule=irrigation_events,
    theta_init=theta_init_val.copy()
)

rmse_adj = np.sqrt(np.mean(
    (val_adj[:,4] - moisture4_real)**2))
print(f"Adjusted params RMSE: {rmse_adj:.4f} cm³/cm³")
print(f"Simulated mean: {val_adj[:,4].mean():.4f} "
      f"(real: {moisture4_real.mean():.4f})")

# Generate 30 days of synthetic data
# with randomized irrigation schedules
N_DAYS = 30
MINS_PER_DAY = 1440
total_minutes = N_DAYS * MINS_PER_DAY

print(f"\nGenerating {N_DAYS} days of synthetic data...")
print(f"Total minutes: {total_minutes}")

# Generate random irrigation schedule
# Based on real data patterns:
# - 2-4 irrigation events per day
# - Each 20-250 minutes long
# - Random times throughout day
# - Rate similar to detected events

def generate_irrigation_schedule(n_days,
                                   min_per_day=1440):
    """Generate realistic random irrigation schedule"""
    events = []
    total_mins = n_days * min_per_day

    t = 0
    while t < total_mins:
        # Random gap between irrigations (4-12 hours)
        gap = random.randint(240, 720)
        t += gap

        if t >= total_mins:
            break

        # Random duration (20-200 minutes)
        duration = random.randint(20, 200)

        # Random irrigation rate (similar to real)
        rate = random.uniform(0.002, 0.004)

        end_t = min(t + duration, total_mins)
        events.append((float(t), float(end_t), rate))

        t = end_t

    return events

synthetic_schedule = generate_irrigation_schedule(N_DAYS)

print(f"Generated {len(synthetic_schedule)} "
      f"irrigation events over {N_DAYS} days")
print(f"Sample events:")
for s, e, r in synthetic_schedule[:5]:
    print(f"  t={s:.0f}-{e:.0f} min "
          f"({(e-s):.0f} min, rate={r:.4f})")

# Initial conditions — start at real data mean
theta_init_syn = np.array([
    moisture4_real.mean() * 2.0,   # moisture0
    moisture4_real.mean() * 3.0,   # moisture1
    moisture4_real.mean() * 4.0,   # moisture2
    moisture4_real.mean() * 1.5,   # moisture3
    moisture4_real.mean()          # moisture4
])
theta_init_syn = np.clip(theta_init_syn,
                          adjusted_params[0] + 0.001,
                          adjusted_params[1] - 0.001)

print(f"\nRunning {N_DAYS}-day simulation...")
print(f"This will take 2-3 minutes...")

synthetic_result = richards_stable_solver(
    adjusted_params,
    n_minutes=total_minutes,
    irrigation_schedule=synthetic_schedule,
    theta_init=theta_init_syn
)

print(f"Simulation complete ✓")
print(f"Output shape: {synthetic_result.shape}")

# Build synthetic DataFrame
# Match real data column structure
time_index = pd.date_range(
    start='2020-03-10 00:00',
    periods=total_minutes,
    freq='1min'
)

# Create irrigation flag
irr_flag = np.zeros(total_minutes, dtype=bool)
for s, e, r in synthetic_schedule:
    irr_flag[int(s):int(e)] = True

df_synthetic = pd.DataFrame({
    'year':       time_index.year,
    'month':      time_index.month,
    'day':        time_index.day,
    'hour':       time_index.hour,
    'minute':     time_index.minute,
    'second':     0,
    'moisture0':  np.clip(synthetic_result[:,0], 0.01, 0.99),
    'moisture1':  np.clip(synthetic_result[:,1], 0.01, 0.99),
    'moisture2':  np.clip(synthetic_result[:,2], 0.01, 0.99),
    'moisture3':  np.clip(synthetic_result[:,3], 0.01, 0.99),
    'moisture4':  np.clip(synthetic_result[:,4], 0.01, 0.99),
    'irrgation':  irr_flag
})

print(f"\nSynthetic DataFrame created:")
print(f"  Shape: {df_synthetic.shape}")
print(f"\nSynthetic moisture4 statistics:")
print(f"  Mean: {df_synthetic['moisture4'].mean():.4f} "
      f"(real: {moisture4_real.mean():.4f})")
print(f"  Std:  {df_synthetic['moisture4'].std():.4f} "
      f"(real: {moisture4_real.std():.4f})")
print(f"  Min:  {df_synthetic['moisture4'].min():.4f} "
      f"(real: {moisture4_real.min():.4f})")
print(f"  Max:  {df_synthetic['moisture4'].max():.4f} "
      f"(real: {moisture4_real.max():.4f})")

# Visualization — compare real vs synthetic
fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Phase 2: Synthetic Data Generation\n'
             'Richards Equation — 30 Days of Synthetic '
             'Soil Moisture Data',
             fontsize=13, fontweight='bold')

# Plot 1: Real data moisture4
time_real = np.arange(len(df_real)) / 60
axes[0].plot(time_real, moisture4_real,
             color='black', linewidth=1.0,
             label='Real moisture4 (4 days)')
for s, e, r in irrigation_events:
    axes[0].axvspan(s/60, e/60, alpha=0.2,
                    color='orange')
axes[0].set_title('Real Data — 4 Days',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('Moisture (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Synthetic data moisture4
time_syn = np.arange(total_minutes) / 60
axes[1].plot(time_syn, df_synthetic['moisture4'].values,
             color='steelblue', linewidth=0.8,
             label='Synthetic moisture4 (30 days)',
             alpha=0.8)
for s, e, r in synthetic_schedule:
    axes[1].axvspan(s/60, e/60, alpha=0.1,
                    color='orange')
axes[1].set_title('Synthetic Data — 30 Days',
                   fontsize=11, fontweight='bold')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Distribution comparison
axes[2].hist(moisture4_real, bins=30,
             alpha=0.6, color='black',
             label='Real moisture4',
             density=True)
axes[2].hist(df_synthetic['moisture4'].values,
             bins=30, alpha=0.6,
             color='steelblue',
             label='Synthetic moisture4',
             density=True)
axes[2].set_title('Distribution Comparison — '
                   'Real vs Synthetic',
                   fontsize=11, fontweight='bold')
axes[2].set_xlabel('Moisture (cm³/cm³)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_data_generation.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Save synthetic data
df_synthetic.to_csv('synthetic_moisture_30days.csv',
                     index=False)
print(f"\nSynthetic data saved: synthetic_moisture_30days.csv")
print(f"Ready for augmented training ✓")

In [ ]:
# ============================================================
# Phase 2 Fix: Rescale synthetic data to match
# real data statistics
# ============================================================

real_mean = moisture4_real.mean()
real_std  = moisture4_real.std()
real_min  = moisture4_real.min()
real_max  = moisture4_real.max()

syn_raw = df_synthetic['moisture4'].values.copy()

# Z-score normalize then rescale to real distribution
syn_normalized = (syn_raw - syn_raw.mean()) / syn_raw.std()
syn_rescaled = syn_normalized * real_std + real_mean

# Clip to real data range
syn_rescaled = np.clip(syn_rescaled, real_min, real_max)

print("Distribution after rescaling:")
print(f"  Mean: {syn_rescaled.mean():.4f} "
      f"(real: {real_mean:.4f})")
print(f"  Std:  {syn_rescaled.std():.4f} "
      f"(real: {real_std:.4f})")
print(f"  Min:  {syn_rescaled.min():.4f} "
      f"(real: {real_min:.4f})")
print(f"  Max:  {syn_rescaled.max():.4f} "
      f"(real: {real_max:.4f})")

# Apply rescaling to all moisture columns
# Use same rescaling approach for all layers
for i, col in enumerate(['moisture0','moisture1',
                          'moisture2','moisture3',
                          'moisture4']):
    raw = df_synthetic[col].values.copy()
    real_col_mean = df_real[col].mean()
    real_col_std  = df_real[col].std()
    real_col_min  = df_real[col].min()
    real_col_max  = df_real[col].max()

    normalized = (raw - raw.mean()) / (raw.std() + 1e-8)
    rescaled = normalized * real_col_std + real_col_mean
    rescaled = np.clip(rescaled, real_col_min, real_col_max)
    df_synthetic[col] = rescaled

print("\nAll columns rescaled to match real distribution ✓")

# Final validation plot
fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Synthetic Data Generation — Final Result\n'
             'Richards Equation Physics + Distribution Rescaling',
             fontsize=13, fontweight='bold')

time_real = np.arange(len(df_real)) / 60
time_syn  = np.arange(len(df_synthetic)) / 60

# Plot 1: Real data
axes[0].plot(time_real, moisture4_real,
             color='black', linewidth=1.0,
             label='Real moisture4 (4 days)')
for s, e, r in irrigation_events:
    axes[0].axvspan(s/60, e/60, alpha=0.2, color='orange')
axes[0].set_title('Real Data — 4 Days',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('Moisture (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Plot 2: Synthetic data
axes[1].plot(time_syn,
             df_synthetic['moisture4'].values,
             color='steelblue', linewidth=0.6,
             label='Synthetic moisture4 (30 days)',
             alpha=0.8)
for s, e, r in synthetic_schedule[:20]:
    axes[1].axvspan(s/60, e/60, alpha=0.1, color='orange')
axes[1].set_title('Synthetic Data — 30 Days '
                   '(Rescaled)',
                   fontsize=11, fontweight='bold')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Distribution comparison
axes[2].hist(moisture4_real, bins=30,
             alpha=0.6, color='black',
             label=f'Real (mean={real_mean:.4f})',
             density=True)
axes[2].hist(df_synthetic['moisture4'].values,
             bins=30, alpha=0.6, color='steelblue',
             label=f'Synthetic (mean='
                   f'{df_synthetic["moisture4"].mean():.4f})',
             density=True)
axes[2].set_title('Distribution Comparison — '
                   'Real vs Synthetic (After Rescaling)',
                   fontsize=11, fontweight='bold')
axes[2].set_xlabel('Moisture (cm³/cm³)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('synthetic_final.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Save rescaled synthetic data
df_synthetic.to_csv('synthetic_moisture_30days_rescaled.csv',
                     index=False)
print("\nRescaled synthetic data saved ✓")
print(f"Shape: {df_synthetic.shape}")
print(f"Ready for augmented LSTM training ✓")

In [ ]:
# ============================================================
# Fix: Increase irrigation rate to generate spikes in m4
# ============================================================

# Test with much higher irrigation rate
test_params_spiky = np.array([
    0.009,   # theta_r
    0.115,   # theta_s
    0.300,   # alpha — much higher for faster wetting
    2.000,   # n — higher for sharper response
    0.050    # Ks — much higher for faster drainage
])

# Very high irrigation rate to force spikes
test_schedule_spiky = [
    (s, e, 0.020)  # 10x higher rate
    for s, e, r in irrigation_events
]

theta_init_test = np.clip(
    df_real[['moisture0','moisture1','moisture2',
             'moisture3','moisture4']].iloc[0].values.copy(),
    test_params_spiky[0] + 0.001,
    test_params_spiky[1] - 0.001
)

test_spiky = richards_stable_solver(
    test_params_spiky,
    n_minutes=len(df_real),
    irrigation_schedule=test_schedule_spiky,
    theta_init=theta_init_test
)

print("Spiky test moisture4 stats:")
print(f"  Mean: {test_spiky[:,4].mean():.4f} "
      f"(real: {moisture4_real.mean():.4f})")
print(f"  Std:  {test_spiky[:,4].std():.4f} "
      f"(real: {moisture4_real.std():.4f})")
print(f"  Min:  {test_spiky[:,4].min():.4f} "
      f"(real: {moisture4_real.min():.4f})")
print(f"  Max:  {test_spiky[:,4].max():.4f} "
      f"(real: {moisture4_real.max():.4f})")

# Quick plot
fig, ax = plt.subplots(figsize=(14, 4))
time_h = np.arange(len(df_real)) / 60
ax.plot(time_h, moisture4_real, color='black',
        linewidth=1.0, label='Real')
ax.plot(time_h, test_spiky[:,4], color='steelblue',
        linewidth=1.5, linestyle='--', label='Simulated')
ax.set_title('Spiky Parameter Test')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Level 1 Statistical Augmentation
# Generate synthetic data by recombining real patterns
# Honest alternative to physics-based generation
# ============================================================

print("="*55)
print("LEVEL 1: STATISTICAL AUGMENTATION")
print("="*55)
print("Generating synthetic data by recombining")
print("real observed patterns with random variation\n")

np.random.seed(42)

# Strategy:
# 1. Extract "wet periods" and "dry periods" from real data
# 2. Recombine them randomly with slight noise
# 3. Generate 30 days of realistic synthetic sequences

# Step 1: Identify wet and dry periods from real data
all_cols = ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']
real_data = df_real[all_cols].values

# Split into segments of 60 minutes
segment_len = 60
n_segments = len(real_data) // segment_len
segments = []

for i in range(n_segments):
    seg = real_data[i*segment_len:(i+1)*segment_len]
    mean_m4 = seg[:, 4].mean()
    segments.append({
        'data': seg.copy(),
        'mean_m4': mean_m4,
        'is_wet': mean_m4 > 0.035
    })

wet_segments = [s for s in segments if s['is_wet']]
dry_segments = [s for s in segments if not s['is_wet']]

print(f"Real data segments: {len(segments)} total")
print(f"  Wet segments (mean m4 > 0.035): {len(wet_segments)}")
print(f"  Dry segments: {len(dry_segments)}")

# Step 2: Generate 30 days by recombining segments
N_DAYS = 30
mins_per_day = 1440
total_mins = N_DAYS * mins_per_day
n_total_segments = total_mins // segment_len

synthetic_segments = []

for i in range(n_total_segments):
    # 15% chance of wet period (similar to real data)
    if np.random.random() < 0.15 and len(wet_segments) > 0:
        seg = np.random.choice(wet_segments)
    else:
        seg = np.random.choice(dry_segments)

    # Add small random noise
    noise_scale = 0.002
    noisy_seg = seg['data'].copy()
    noisy_seg += np.random.normal(0, noise_scale,
                                    noisy_seg.shape)

    # Clip to real data range
    for j, col in enumerate(all_cols):
        noisy_seg[:, j] = np.clip(
            noisy_seg[:, j],
            df_real[col].min(),
            df_real[col].max()
        )

    synthetic_segments.append(noisy_seg)

# Concatenate all segments
synthetic_array = np.vstack(synthetic_segments)
print(f"\nSynthetic array shape: {synthetic_array.shape}")

# Step 3: Create smooth transitions between segments
# Apply light smoothing to remove abrupt jumps
from scipy.ndimage import uniform_filter1d
for j in range(synthetic_array.shape[1]):
    synthetic_array[:, j] = uniform_filter1d(
        synthetic_array[:, j], size=3)
    synthetic_array[:, j] = np.clip(
        synthetic_array[:, j],
        df_real[all_cols[j]].min(),
        df_real[all_cols[j]].max()
    )

# Step 4: Build DataFrame
time_index = pd.date_range(
    start='2020-03-10 00:00',
    periods=total_mins,
    freq='1min'
)

# Create irrigation flag from moisture spikes
irr_flag = synthetic_array[:, 1] > 0.60

df_synthetic_l1 = pd.DataFrame({
    'year':      time_index.year,
    'month':     time_index.month,
    'day':       time_index.day,
    'hour':      time_index.hour,
    'minute':    time_index.minute,
    'second':    0,
    'moisture0': synthetic_array[:, 0],
    'moisture1': synthetic_array[:, 1],
    'moisture2': synthetic_array[:, 2],
    'moisture3': synthetic_array[:, 3],
    'moisture4': synthetic_array[:, 4],
    'irrgation': irr_flag
})

print(f"\nSynthetic DataFrame shape: {df_synthetic_l1.shape}")
print(f"\nDistribution comparison:")
for col in all_cols:
    print(f"  {col}:")
    print(f"    Real:      mean={df_real[col].mean():.4f} "
          f"std={df_real[col].std():.4f}")
    print(f"    Synthetic: mean={df_synthetic_l1[col].mean():.4f} "
          f"std={df_synthetic_l1[col].std():.4f}")

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(18, 14))
fig.suptitle('Level 1 Statistical Augmentation\n'
             '30 Days Synthetic Data via Pattern Recombination',
             fontsize=13, fontweight='bold')

time_real = np.arange(len(df_real)) / 60
time_syn = np.arange(total_mins) / 60

axes[0].plot(time_real,
             df_real['moisture4'].values,
             color='black', linewidth=1.0,
             label='Real moisture4 (4 days)')
axes[0].set_title('Real Data — 4 Days',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('Moisture (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(time_syn,
             df_synthetic_l1['moisture4'].values,
             color='steelblue', linewidth=0.6,
             label='Synthetic moisture4 (30 days)',
             alpha=0.8)
axes[1].set_title('Synthetic Data — 30 Days '
                   '(Statistical Augmentation)',
                   fontsize=11, fontweight='bold')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[2].hist(df_real['moisture4'].values, bins=30,
             alpha=0.6, color='black',
             label=f'Real (mean='
                   f'{df_real["moisture4"].mean():.4f})',
             density=True)
axes[2].hist(df_synthetic_l1['moisture4'].values,
             bins=30, alpha=0.6, color='steelblue',
             label=f'Synthetic (mean='
                   f'{df_synthetic_l1["moisture4"].mean():.4f})',
             density=True)
axes[2].set_title('Distribution Comparison',
                   fontsize=11, fontweight='bold')
axes[2].set_xlabel('Moisture (cm³/cm³)')
axes[2].set_ylabel('Density')
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('statistical_augmentation.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Save
df_synthetic_l1.to_csv(
    'synthetic_moisture_statistical.csv', index=False)
print("\nStatistical augmentation complete ✓")
print("synthetic_moisture_statistical.csv saved ✓")

In [ ]:
# ============================================================
# Sensor Quantization Fix
# Real sensor reports discrete 0.01 cm³/cm³ steps
# Apply same quantization to synthetic data
# Prevents domain shift between synthetic and real
# ============================================================

print("Applying sensor quantization to synthetic data...")
print(f"Before quantization:")
print(f"  moisture4 unique values (sample): "
      f"{sorted(df_synthetic_l1['moisture4'].unique())[:10]}")

quantization_step = 0.01

# Apply quantization to all moisture columns
for col in ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']:
    df_synthetic_l1[col] = (
        np.round(df_synthetic_l1[col] /
                 quantization_step) * quantization_step
    )

print(f"\nAfter quantization:")
print(f"  moisture4 unique values: "
      f"{sorted(df_synthetic_l1['moisture4'].unique())}")
print(f"\nDistribution after quantization:")
print(f"  moisture4 mean: {df_synthetic_l1['moisture4'].mean():.4f} "
      f"(real: {moisture4_real.mean():.4f})")
print(f"  moisture4 std:  {df_synthetic_l1['moisture4'].std():.4f} "
      f"(real: {moisture4_real.std():.4f})")

# Resave with quantization applied
df_synthetic_l1.to_csv(
    'synthetic_moisture_final.csv', index=False)
print(f"\nFinal synthetic data saved: "
      f"synthetic_moisture_final.csv ✓")
print(f"Shape: {df_synthetic_l1.shape}")
print(f"Quantization applied: {quantization_step} cm³/cm³ ✓")

In [ ]:
# ============================================================
# Noise Analysis — Real vs Synthetic
# Check if synthetic noise matches real sensor artifacts
# ============================================================

print("="*55)
print("NOISE CHARACTERISTICS ANALYSIS")
print("="*55)

# Real data noise — compute point-to-point variation
real_m4 = df_real['moisture4'].values
real_noise = np.diff(real_m4)

# Synthetic data noise
syn_m4 = df_synthetic_l1['moisture4'].values
syn_noise = np.diff(syn_m4)

print("\nReal moisture4 noise characteristics:")
print(f"  Point-to-point std: {real_noise.std():.5f}")
print(f"  Point-to-point mean: {real_noise.mean():.5f}")
print(f"  Max jump: {np.abs(real_noise).max():.5f}")
print(f"  % timesteps with zero change: "
      f"{(real_noise==0).mean()*100:.1f}%")

print("\nSynthetic moisture4 noise characteristics:")
print(f"  Point-to-point std: {syn_noise.std():.5f}")
print(f"  Point-to-point mean: {syn_noise.mean():.5f}")
print(f"  Max jump: {np.abs(syn_noise).max():.5f}")
print(f"  % timesteps with zero change: "
      f"{(syn_noise==0).mean()*100:.1f}%")

# Plot noise comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Noise Characteristics — Real vs Synthetic',
             fontsize=13, fontweight='bold')

axes[0].hist(real_noise, bins=50,
             color='black', alpha=0.7,
             label='Real noise', density=True)
axes[0].hist(syn_noise, bins=50,
             color='steelblue', alpha=0.7,
             label='Synthetic noise', density=True)
axes[0].set_title('Point-to-Point Variation Distribution')
axes[0].set_xlabel('Change per timestep (cm³/cm³)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Zoomed time series comparison
axes[1].plot(real_m4[:200], color='black',
             linewidth=1.0, label='Real (200 samples)')
axes[1].plot(syn_m4[:200], color='steelblue',
             linewidth=1.0, label='Synthetic (200 samples)',
             alpha=0.8)
axes[1].set_title('Visual Texture Comparison\n'
                   'First 200 Timesteps')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('noise_analysis.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Noise Enhancement Fix
# Make synthetic noise match real sensor characteristics
# ============================================================

print("Enhancing synthetic noise to match real sensor...")

# Reload clean synthetic data before quantization
# We need to add noise BEFORE quantizing
syn_enhanced = df_synthetic_l1.copy()

for col in ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']:

    real_col = df_real[col].values
    real_noise_std = np.diff(real_col).std()

    # Current synthetic values
    syn_col = syn_enhanced[col].values.copy()

    # Add noise scaled to match real sensor
    noise = np.random.normal(0, real_noise_std * 0.8,
                              len(syn_col))
    syn_col = syn_col + noise

    # Occasionally add larger jumps (5% of timesteps)
    # matching real sensor's sudden changes
    jump_mask = np.random.random(len(syn_col)) < 0.05
    jump_sizes = np.random.choice(
        [-0.01, 0.01, -0.02, 0.02],
        size=jump_mask.sum()
    )
    syn_col[jump_mask] += jump_sizes

    # Clip to real data range
    syn_col = np.clip(syn_col,
                       df_real[col].min(),
                       df_real[col].max())

    syn_enhanced[col] = syn_col

# Reapply quantization after noise
quantization_step = 0.01
for col in ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']:
    syn_enhanced[col] = (
        np.round(syn_enhanced[col] /
                 quantization_step) * quantization_step
    )

# Recheck noise characteristics
real_noise_check = np.diff(df_real['moisture4'].values)
syn_noise_check = np.diff(syn_enhanced['moisture4'].values)

print(f"\nAfter enhancement:")
print(f"Real  std: {real_noise_check.std():.5f} | "
      f"Synthetic std: {syn_noise_check.std():.5f}")
print(f"Real  max: {np.abs(real_noise_check).max():.5f} | "
      f"Synthetic max: {np.abs(syn_noise_check).max():.5f}")
print(f"Real  zero%: {(real_noise_check==0).mean()*100:.1f}% | "
      f"Synthetic zero%: {(syn_noise_check==0).mean()*100:.1f}%")

print(f"\nDistribution after enhancement:")
print(f"  moisture4 mean: {syn_enhanced['moisture4'].mean():.4f} "
      f"(real: {df_real['moisture4'].mean():.4f})")
print(f"  moisture4 std:  {syn_enhanced['moisture4'].std():.4f} "
      f"(real: {df_real['moisture4'].std():.4f})")

# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Enhanced Noise — Real vs Synthetic',
             fontsize=13, fontweight='bold')

axes[0].hist(real_noise_check, bins=50,
             color='black', alpha=0.7,
             label='Real noise', density=True)
axes[0].hist(syn_noise_check, bins=50,
             color='steelblue', alpha=0.7,
             label='Enhanced synthetic', density=True)
axes[0].set_title('Point-to-Point Variation')
axes[0].set_xlabel('Change per timestep (cm³/cm³)')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_real['moisture4'].values[:200],
             color='black', linewidth=1.0,
             label='Real (200 samples)')
axes[1].plot(syn_enhanced['moisture4'].values[:200],
             color='steelblue', linewidth=1.0,
             label='Enhanced synthetic',
             alpha=0.8)
axes[1].set_title('Visual Texture Comparison')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('noise_enhanced.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Save enhanced version
syn_enhanced.to_csv('synthetic_moisture_final.csv',
                     index=False)
print(f"\nEnhanced synthetic data saved ✓")
print(f"synthetic_moisture_final.csv updated")

In [ ]:
# ============================================================
# Noise Fix v2 — Better calibrated
# ============================================================

syn_enhanced_v2 = df_synthetic_l1.copy()

for col in ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']:

    real_col = df_real[col].values
    real_noise_std = np.diff(real_col).std()
    real_zero_pct = (np.diff(real_col) == 0).mean()

    syn_col = syn_enhanced_v2[col].values.copy()

    # Much smaller noise — only 20% of real noise std
    noise = np.random.normal(0, real_noise_std * 0.20,
                              len(syn_col))

    # Only apply noise to 20% of timesteps
    # matching real sensor's mostly-flat behavior
    noise_mask = np.random.random(len(syn_col)) > real_zero_pct
    final_noise = np.zeros(len(syn_col))
    final_noise[noise_mask] = noise[noise_mask]

    syn_col = syn_col + final_noise

    # Rare large jumps — 1% of timesteps only
    jump_mask = np.random.random(len(syn_col)) < 0.01
    jump_sizes = np.random.choice(
        [-0.01, 0.01, -0.02, 0.02],
        size=jump_mask.sum()
    )
    syn_col[jump_mask] += jump_sizes

    # Clip to real range
    syn_col = np.clip(syn_col,
                       df_real[col].min(),
                       df_real[col].max())

    syn_enhanced_v2[col] = syn_col

# Reapply quantization
quantization_step = 0.01
for col in ['moisture0','moisture1','moisture2',
            'moisture3','moisture4']:
    syn_enhanced_v2[col] = (
        np.round(syn_enhanced_v2[col] /
                 quantization_step) * quantization_step
    )

# Check results
real_noise_check = np.diff(df_real['moisture4'].values)
syn_noise_check = np.diff(syn_enhanced_v2['moisture4'].values)

print("Noise calibration v2:")
print(f"  Std:   Real={real_noise_check.std():.5f} | "
      f"Synthetic={syn_noise_check.std():.5f}")
print(f"  Max:   Real={np.abs(real_noise_check).max():.5f} | "
      f"Synthetic={np.abs(syn_noise_check).max():.5f}")
print(f"  Zero%: Real={((real_noise_check==0).mean()*100):.1f}% | "
      f"Synthetic={((syn_noise_check==0).mean()*100):.1f}%")
print(f"\n  moisture4 mean: "
      f"{syn_enhanced_v2['moisture4'].mean():.4f} "
      f"(real: {df_real['moisture4'].mean():.4f})")
print(f"  moisture4 std:  "
      f"{syn_enhanced_v2['moisture4'].std():.4f} "
      f"(real: {df_real['moisture4'].std():.4f})")

# Visual check
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Noise Calibration v2 — Real vs Synthetic',
             fontsize=13, fontweight='bold')

axes[0].hist(real_noise_check, bins=50,
             color='black', alpha=0.7,
             label='Real', density=True)
axes[0].hist(syn_noise_check, bins=50,
             color='steelblue', alpha=0.7,
             label='Synthetic v2', density=True)
axes[0].set_title('Point-to-Point Variation')
axes[0].set_xlabel('Change per timestep')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_real['moisture4'].values[:200],
             color='black', linewidth=1.0,
             label='Real')
axes[1].plot(syn_enhanced_v2['moisture4'].values[:200],
             color='steelblue', linewidth=1.0,
             label='Synthetic v2', alpha=0.8)
axes[1].set_title('Visual Texture v2')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Moisture (cm³/cm³)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('noise_calibrated.png', dpi=150,
            bbox_inches='tight')
plt.show()

# Save
syn_enhanced_v2.to_csv('synthetic_moisture_final.csv',
                        index=False)
print("\nCalibrated synthetic data saved ✓")

In [ ]:
# ============================================================
# Combine Real + Synthetic Data
# Prepare for chronological split evaluation
# ============================================================

print("="*55)
print("COMBINING REAL + SYNTHETIC DATASETS")
print("="*55)

# Load both datasets
df_real_final = df_real.copy()
df_synthetic_final = syn_enhanced_v2.copy()

print(f"Real data:      {len(df_real_final):,} samples "
      f"({len(df_real_final)/60/24:.1f} days)")
print(f"Synthetic data: {len(df_synthetic_final):,} samples "
      f"({len(df_synthetic_final)/60/24:.1f} days)")

# Concatenate — real first then synthetic
df_combined = pd.concat([df_real_final,
                          df_synthetic_final],
                          ignore_index=True)

print(f"Combined:       {len(df_combined):,} samples "
      f"({len(df_combined)/60/24:.1f} days)")

# Verify irrigation events across full timeline
moisture4_combined = df_combined['moisture4'].values
spike_indices = np.where(moisture4_combined > 0.04)[0]

print(f"\nIrrigation spikes (>0.04) in combined data:")
print(f"  Total spike timesteps: {len(spike_indices)}")
print(f"  First spike at: sample {spike_indices[0]} "
      f"(hour {spike_indices[0]/60:.1f})")
print(f"  Last spike at: sample {spike_indices[-1]} "
      f"(hour {spike_indices[-1]/60:.1f})")

# Check distribution across timeline
# Divide into 8 equal chunks and count spikes in each
n_chunks = 8
chunk_size = len(df_combined) // n_chunks
print(f"\nSpike distribution across {n_chunks} equal chunks:")
for i in range(n_chunks):
    start = i * chunk_size
    end = (i+1) * chunk_size
    chunk_spikes = (moisture4_combined[start:end] > 0.04).sum()
    pct = chunk_spikes / chunk_size * 100
    print(f"  Chunk {i+1} "
          f"(hours {start/60:.0f}-{end/60:.0f}): "
          f"{chunk_spikes} spikes ({pct:.1f}%)")

# Verify chronological split will work
split_idx_combined = int(len(df_combined) * 0.8)
train_spikes = (moisture4_combined[:split_idx_combined] > 0.04).sum()
test_spikes = (moisture4_combined[split_idx_combined:] > 0.04).sum()

print(f"\nChronological 80/20 split preview:")
print(f"  Train: {split_idx_combined:,} samples "
      f"— {train_spikes} spikes ✓")
print(f"  Test:  {len(df_combined)-split_idx_combined:,} samples "
      f"— {test_spikes} spikes ✓")

if test_spikes > 0:
    print(f"\n✅ Test set contains irrigation events!")
    print(f"   Chronological split is now VALID")
else:
    print(f"\n❌ Test set still has no spikes")
    print(f"   Need more synthetic data")

# Full timeline visualization
fig, axes = plt.subplots(2, 1, figsize=(18, 8))
fig.suptitle('Combined Dataset — Real + Synthetic\n'
             'Verifying Irrigation Event Distribution',
             fontsize=13, fontweight='bold')

time_hours = np.arange(len(df_combined)) / 60

axes[0].plot(time_hours, moisture4_combined,
             color='steelblue', linewidth=0.5,
             alpha=0.7)
axes[0].axvline(x=split_idx_combined/60,
                color='red', linewidth=2,
                linestyle='--',
                label=f'80/20 split point '
                      f'(hour {split_idx_combined/60:.0f})')
axes[0].axvspan(0, len(df_real)/60,
                alpha=0.1, color='green',
                label='Real data (4 days)')
axes[0].axvspan(len(df_real)/60,
                len(df_combined)/60,
                alpha=0.1, color='orange',
                label='Synthetic data (30 days)')
axes[0].set_title('Full Combined Timeline with Split Point')
axes[0].set_ylabel('moisture4 (cm³/cm³)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Distribution comparison
axes[1].hist(moisture4_combined[:split_idx_combined],
             bins=30, alpha=0.6, color='steelblue',
             label=f'Train set ({train_spikes} spikes)',
             density=True)
axes[1].hist(moisture4_combined[split_idx_combined:],
             bins=30, alpha=0.6, color='darkorange',
             label=f'Test set ({test_spikes} spikes)',
             density=True)
axes[1].set_title('Train vs Test Distribution — '
                   'Combined Dataset')
axes[1].set_xlabel('moisture4 (cm³/cm³)')
axes[1].set_ylabel('Density')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('combined_dataset.png', dpi=150,
            bbox_inches='tight')
plt.show()